# U-Net r34 at native 2048 px (Charan)

Same recipe as U-Net run 2 (7-day split, honest validation), but at the competition's native resolution. The inputs are prepared at 2048 px, training uses random 1024 px crops (the same cost per step as the 1024 px runs), and inference runs on the whole 2048 px image. That gives thin filaments and barbs 2x the pixels. Compare it with run 2 on the same split: val PQ 0.384 with the new keep rule.

At the end, `scripts/ensemble.py` sweeps the keep rule for this model on the 7-day val split (D4 TTA) and writes `submission_u2048_pq.csv`.

**Settings:** GPU T4 x2, Internet on, competition data.

In [ ]:
# 1. Setup
import os, subprocess, sys, json, shutil
from pathlib import Path
import pandas as pd

BRANCH = "charan/ensemble"
REPO = "https://github.com/hsfcoulibaly/aml_project.git"
SRC = Path("/kaggle/working/aml_project")
if not SRC.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(SRC)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "pycocotools"], check=True)
CODE = SRC / "Charan" / "unet"
os.chdir(CODE); sys.path.insert(0, str(CODE))
os.environ["UNET_WORK_DIR"] = "/kaggle/working/unet"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORK = Path(os.environ["UNET_WORK_DIR"])
OUT = Path("/kaggle/working/unet_results"); OUT.mkdir(exist_ok=True)
print(subprocess.run(["git", "-C", str(SRC), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Prepare native 2048 px inputs on the 7-day split (~10 min; ~5 GB on disk)
!python scripts/prepare.py --size 2048 --workers 4 --split splits/week_split.json
!du -sh {WORK}/images {WORK}/masks

In [ ]:
# 3. Train on 1024 px crops, validate on whole 2048 px images (~3 h)
RUN = "unet_r34_2048_week"
!python scripts/train.py --name {RUN} --encoder resnet34 --crop 1024 --epochs 40 --batch 8 --lr 3e-4 --pos-weight 10 --pq-every 2 --patience 10 --workers 4

In [ ]:
h = pd.read_csv(WORK / "runs" / RUN / "history.csv")
display(h.tail(8))

In [ ]:
# 4. Sweep the keep rule on val (honest) and write the test submission
!python scripts/ensemble.py --model u2048={WORK}/runs/{RUN}/best.pt --tta d4 --batch 1 --workers 4 --out-dir {OUT}/ensemble

In [ ]:
# Keep weights, history, sweeps and CSVs; drop the large prepared data
for f in ["best.pt", "history.csv", "args.json"]:
    p = WORK / "runs" / RUN / f
    if p.exists(): shutil.copy(p, OUT / f)
for p in (OUT / "ensemble").iterdir(): print(p.name)
print(json.dumps(json.load(open(OUT / "ensemble" / "summary.json")), indent=1)[:4000])
shutil.rmtree(WORK / "images", ignore_errors=True); shutil.rmtree(WORK / "masks", ignore_errors=True)
(WORK / "runs" / RUN / "last_state.pt").unlink(missing_ok=True)
shutil.rmtree(SRC, ignore_errors=True)